In [1]:
def is_noise_title(title):
    t = (title or "").lower().strip()

    noise_patterns = [
        "introduction",
        "document structure",
        "structure of",
        "overview",
        "appendix",
        "table of contents",
        "foreword",
        "preface",
        "figure",
        "fig",
        "acronym",
        "glossary",
        "table"
    ]

    return any(pattern in t for pattern in noise_patterns)

In [ ]:
import json

combined_rows = []

# -----------------------------
# LOAD DATA
# -----------------------------
with open(r"D:\exercises\Thesis\nist_sp_800.json", "r", encoding="utf-8") as f:
    nist_data = json.load(f)

with open(r"D:\exercises\Thesis\bsi_standard.json", "r", encoding="utf-8") as f:
    bsi_data = json.load(f)


# -----------------------------
# BSI (already structured)
# -----------------------------
for item in bsi_data:

    if is_noise_title(item.get("Title", "")):
        continue

    combined_rows.append({
        "source_standard": "BSI",
        "source_id": item.get("source_id"),
        "section_number": item.get("section_number"),
        "chunk_id": f"BSI-{item.get('chunk_id')}",

        "title": item.get("title"),

        "content": item.get("content"),
        "summary": item.get("summary"),

        "lifecycle_phase": item.get("lifecycle_phase"),
        "normative": item.get("normative"),
        "normative_type": item.get("normative_type"),

        "keywords": item.get("keywords") or [],
        "abstraction_level": "governance"
    })


# -----------------------------
# NIST (already chunked)
# -----------------------------
for item in nist_data:

    if is_noise_title(item.get("title", "")):
        continue

    combined_rows.append({
        "source_standard": "NIST",
        "source_id": item.get("source_id"),
        "section_number": item.get("section_number"),
        "chunk_id": f"NIST-{item.get('chunk_id')}",

        "title": item.get("title"),
        "content": item.get("content"),
        "summary": item.get("summary"),

        "lifecycle_phase": item.get("lifecycle_phase"),
        "normative": item.get("normative"),
        "normative_type": item.get("normative_type"),

        "keywords": item.get("keywords") or [],
        "abstraction_level": "technical"
    })


# -----------------------------
# SAVE COMBINED FILE
# -----------------------------
with open(r"D:\exercises\Thesis\combined_dataset.json", "w", encoding="utf-8") as f:
    json.dump(combined_rows, f, ensure_ascii=False, indent=2)

print(f" Combined dataset created: {len(combined_rows)} rows")

✅ Combined dataset created: 1011 rows


In [5]:
combined_rows[0]  # Display the first combined row for verification

{'source_standard': 'BSI',
 'source_id': 'BSI-1.4',
 'section_number': '1.4',
 'chunk_id': 'BSI-1.4_0',
 'title': 'Application',
 'content': 'The present standard describes how an information security management system (ISMS) can be designed. A management system encompasses all the provisions ensuring the supervision and management so that the organisation can achieve its objectives.',
 'summary': 'The present standard describes how an information security management system (ISMS) can be designed. A management system encompasses all the provisions ensuring the supervision and management so that the organisation can achieve its objectives.',
 'lifecycle_phase': 'Do',
 'normative': True,
 'normative_type': 'shall',
 'keywords': ['security', 'management', 'information', 'standard', 'system'],
 'abstraction_level': 'governance',
 'embedding': [0.030633527785539627,
  0.003635266562923789,
  -0.056542761623859406,
  -0.034853365272283554,
  -0.030440980568528175,
  -0.0001709621137706563,
 

## Create Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

#model = SentenceTransformer('all-mpnet-base-v2')

def create_embedding(rows):
   # text = f'{row["title"]}{row["content"]} {row["summary"]}'

    text = f"""
            {row['title']}
            {row['summary']}{row['content']}
            {row['keywords']}{row['lifecycle_phase']} {row['normative_type']}
        """
    return model.encode(text).tolist()

for row in combined_rows:
    row["embedding"] = create_embedding(row)



c:\Users\dipan\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3097.58it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
import requests


SUPABASE_URL = "https://kaueqrfosfzpdhtnwdko.supabase.co"
SUPABASE_KEY = "sb_publishable_xb-nsEG892PB_8CgyulR0w_T1MVv2Z9"

table = 'requirements_combined'

url = f"{SUPABASE_URL}/rest/v1/{table}"


headers = {
    "apikey": SUPABASE_KEY,
    "Authorization": f"Bearer {SUPABASE_KEY}",
    "Content-Type": "application/json"
}

delete_response = requests.delete(url + "?id=neq.0", headers=headers)
print("DELETE:", delete_response.status_code)

response = requests.post(url, headers=headers, json=combined_rows)

print(response.status_code)
print(response.text)

DELETE: 204
201



In [8]:
bsi_rows = [r for r in combined_rows if r["source_standard"] == "BSI"]
nist_rows = [r for r in combined_rows if r["source_standard"] == "NIST"]

### Update Cosine Similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

lifecycle_map = {
    "Plan": "Identify",
    "Do": "Protect",
    "Check": "Detect",
    "Act": "Respond"
}

mappings = []

for bsi in bsi_rows:
    best_match = None
    best_score = 0
    best_lifecycle_match = False
    if "appendix" in (bsi.get("title") or "").lower():
        continue
    bsi_phase = bsi.get("lifecycle_phase")

    for nist in nist_rows:
        if "document structure" in (nist.get("title") or "").lower():
            continue
        nist_phase = nist.get("lifecycle_phase")

        sim = cosine_similarity(
            [bsi["embedding"]],
            [nist["embedding"]]
        )[0][0]

        # lifecycle bonus (existing)
        lifecycle_match = False
        if bsi_phase in lifecycle_map:
            if lifecycle_map[bsi_phase] == nist_phase:
                lifecycle_match = True
                sim += 0.05

        
        content = (nist.get("content") or "").lower()
        if any(word in content for word in ["shall", "should", "must"]):
            sim += 0.05

        # keep best match
        if sim > best_score:
            best_score = sim
            best_match = nist
            best_lifecycle_match = lifecycle_match

    if best_match:
        mappings.append({
            "bsi_source_id": bsi["source_id"],
            "nist_source_id": best_match["source_id"],
            "similarity_score": float(best_score),
            "lifecycle_match": best_lifecycle_match
        })

In [10]:
import requests

url = f"{SUPABASE_URL}/rest/v1/requirements_combined"

headers = {
    "apikey": SUPABASE_KEY,
    "Authorization": f"Bearer {SUPABASE_KEY}"
}

response = requests.get(url, headers=headers)
rows = response.json()

In [11]:
id_map = {row["source_id"]: row["id"] for row in rows}
id_map

{'BSI-1.4': 8,
 'BSI-2.1.1': 21,
 'BSI-3.1': 25,
 'BSI-3': 32,
 'BSI-4': 38,
 'BSI-5': 58,
 'BSI-4.3': 49,
 'NIST-2.3.1': 146,
 'BSI-6': 65,
 'BSI-7': 79,
 'BSI-7.1': 74,
 'BSI-7.4': 87,
 'BSI-7.5': 89,
 'BSI-8': 106,
 'BSI-8.2': 110,
 'BSI-8.4': 112,
 'NIST-2': 123,
 'NIST-5': 307,
 'NIST-9': 729,
 'NIST-10': 731,
 'NIST-2.3.2': 151,
 'NIST-14': 737,
 'NIST-29': 767,
 'NIST-15': 154,
 'NIST-2.3.3': 162,
 'NIST-2.3.4': 165,
 'NIST-2.3.5': 168,
 'NIST-2.3.7': 181,
 'NIST-2.3.6': 175,
 'NIST-2.3.8': 187,
 'NIST-28': 765,
 'NIST-31': 771,
 'NIST-32': 772,
 'NIST-33': 240,
 'NIST-3.2.4': 264,
 'NIST-3.2.1': 246,
 'NIST-3.2.2': 248,
 'NIST-3.2.3': 250,
 'NIST-3.3.1': 267,
 'NIST-3.3.2': 274,
 'NIST-4': 275,
 'NIST-40': 277,
 'NIST-3.3.4': 280,
 'NIST-3.3.5': 282,
 'NIST-3.3.6': 285,
 'NIST-3.3.7': 287,
 'NIST-42': 288,
 'NIST-3.3.8': 292,
 'NIST-3.3.9': 297,
 'NIST-3.3.10': 298,
 'NIST-186': 300,
 'NIST-44': 306,
 'NIST-6': 308,
 'NIST-45': 312,
 'NIST-46': 313,
 'NIST-4.1.1': 315,
 'NIST-4

In [12]:
db_mappings = []
# Source id is the BSI and target id is the NIST. We need to map these to the actual database IDs for the unified table.
for m in mappings:
    source_id = id_map.get(m["bsi_source_id"])
    target_id = id_map.get(m["nist_source_id"])

    if source_id and target_id:
        db_mappings.append({
            "source_requirement_id": source_id,
            "target_requirement_id": target_id,
            "similarity_score": m["similarity_score"],
            "mapping_type": "embedding+lifecycle"
        })

In [13]:
mapping_url = f"{SUPABASE_URL}/rest/v1/requirement_mappings"

headers = {
    "apikey": SUPABASE_KEY,
    "Authorization": f"Bearer {SUPABASE_KEY}",
    "Content-Type": "application/json"
}
delete_response = requests.delete(mapping_url + "?id=neq.0", headers=headers)
print("DELETE:", delete_response.status_code)
response = requests.post(mapping_url, headers=headers, json=db_mappings)

print(response.status_code)
print(response.text)

DELETE: 204
201



In [14]:
print(db_mappings[:2])

[{'source_requirement_id': 8, 'target_requirement_id': 459, 'similarity_score': 0.7081460747836057, 'mapping_type': 'embedding+lifecycle'}, {'source_requirement_id': 8, 'target_requirement_id': 264, 'similarity_score': 0.7647938432332158, 'mapping_type': 'embedding+lifecycle'}]


In [15]:
print(bsi["source_id"], best_score)
print(nist["source_id"], best_score)

BSI-8.4 0.7202127942710811
NIST-298 0.7202127942710811


In [ ]:
import openai

from config.config import OPENAI_API_KEY

# ---------------------------------------------------
# OPENAI CLIENT
# ---------------------------------------------------
client = openai.OpenAI(
    api_key=OPENAI_API_KEY
)

# ---------------------------------------------------
# CREATE EMBEDDING
# ---------------------------------------------------
def create_embedding(row):

    text = f"""
    {row['title']}
    {row['summary']}
    {row['content']}
    {row['keywords']}
    {row['lifecycle_phase']}
    {row['normative_type']}
    """

    response = client.embeddings.create(
        input=text,
        model="text-embedding-3-small"
    )

    return response.data[0].embedding


# ---------------------------------------------------
# TEST ON FIRST ROW
# ---------------------------------------------------
test_row = combined_rows[0]

embedding = create_embedding(test_row)

print("Embedding length:", len(embedding))

print(type(embedding))

print(embedding[:5])